In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import os
import csv
import glob

# Set device to CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------
# 10-Word Vocabulary (index 0 reserved for <blank> / EOS)
# Must match the training app's word list exactly.
VOCAB = [
    "<blank>",  # 0 - EOS / blank token
    "yes",      # 1
    "no",       # 2
    "help",     # 3
    "stop",     # 4
    "go",       # 5
    "water",    # 6
    "food",     # 7
    "pain",     # 8
    "more",     # 9
    "need",     # 10
]
VOCAB_SIZE = len(VOCAB)  # 11
EOS_TOKEN = 0  # blank / end-of-sequence token

# Maps word name (uppercase, as stored in CSV) to vocab index
WORD_NAME_TO_ID = {
    "YES": 1, "NO": 2, "HELP": 3, "STOP": 4, "GO": 5,
    "WATER": 6, "FOOD": 7, "PAIN": 8, "MORE": 9, "NEED": 10,
}

# Training pairs (app word_id → vocab index)
# App word_id 0=YES, 1=NO, 2=HELP, 3=STOP, 4=GO, 5=WATER, 6=FOOD, 7=PAIN, 8=MORE, 9=NEED
APP_ID_TO_VOCAB = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10}

# All single-word training targets (each word is a class to predict)
coherent_fragments = [[i] for i in range(1, VOCAB_SIZE)]  # [[1], [2], ..., [10]]

# Test targets
new_target_sentences = [[i] for i in range(1, VOCAB_SIZE)]


# ------------------------------
# EEGNet-inspired feature extractor (output sequence length = 32)
class EEGNetFeatureExtractor(nn.Module):
    def __init__(self, Chans=16, Samples=256, dropoutRate=0.2, kernLength=64, F1=8, D=2, F2=16):
        super(EEGNetFeatureExtractor, self).__init__()
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kernLength), padding=(0, kernLength//2), bias=False),
            nn.BatchNorm2d(F1, affine=False)
        )
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(F1, F1*D, kernel_size=(Chans, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D, affine=False),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),  # 256 -> 64
            nn.Dropout(dropoutRate)
        )
        self.separableConv = nn.Sequential(
            nn.Conv2d(F1*D, F2, kernel_size=(1, 8), padding=(0, 8//2), bias=False),
            nn.BatchNorm2d(F2, affine=False),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 2)),  # 64 -> 32
            nn.Dropout(dropoutRate)
        )

    def forward(self, x):
        x = x.unsqueeze(1)       # (batch, 1, Chans, Samples)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)  # (batch, F2, 1, 32)
        x = x.squeeze(2)           # (batch, F2, 32)
        x = x.permute(0, 2, 1)     # (batch, 32, F2)
        return x


# ------------------------------
# Sequence model: EEG -> LSTM -> Classifier
class EEGToTextModel(nn.Module):
    def __init__(self, hidden_dim=128, vocab_size=VOCAB_SIZE, num_layers=2):
        super(EEGToTextModel, self).__init__()
        self.feature_extractor = EEGNetFeatureExtractor(Chans=16, Samples=256)
        self.lstm = nn.LSTM(input_size=16, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim*2, vocab_size)

    def forward(self, eeg_input):
        features = self.feature_extractor(eeg_input)  # (batch, 32, 16)
        lstm_out, _ = self.lstm(features)              # (batch, 32, hidden_dim*2)
        logits = self.classifier(lstm_out)             # (batch, 32, vocab_size)
        return logits


# ------------------------------
# Epsilon-greedy decoder
def epsilon_greedy_decode(logits, epsilon=0.0, target_seq=None, max_len=None, repetition_limit=2):
    probs = torch.softmax(logits, dim=-1)
    seq_len = probs.size(1)
    if max_len is None:
        max_len = len(target_seq) if target_seq is not None else seq_len
    decoded = []
    last_token = None
    repeat_count = 0
    for t in range(seq_len):
        if random.random() < epsilon:
            token = random.randint(1, VOCAB_SIZE - 1)
        else:
            token = torch.argmax(probs[0, t]).item()
        if token == EOS_TOKEN:
            break
        if token == last_token:
            repeat_count += 1
            if repeat_count >= repetition_limit:
                break
        else:
            repeat_count = 1
        decoded.append(token)
        last_token = token
        if len(decoded) >= max_len:
            break
    return decoded


# ------------------------------
# Simulated EEG Dataset (used for pre-training before real data is available)
class EEGDataset(Dataset):
    def __init__(self, num_samples=200):
        self.num_samples = num_samples
        self.channels = 16
        self.time_samples = 256
        self.data = []
        self.labels = []
        np.random.seed(42)
        # Fixed EEG pattern for each token
        self.token_patterns = {
            token: torch.tensor(np.random.randn(self.channels, self.time_samples//4), dtype=torch.float).to(device)
            for token in range(1, VOCAB_SIZE)
        }
        for _ in range(num_samples):
            label_seq = random.choice(coherent_fragments)
            self.labels.append(torch.tensor(label_seq, dtype=torch.long).to(device))
            eeg = torch.randn(self.channels, self.time_samples, device=device) * 0.1
            seq_length = len(label_seq)
            seg_len = self.time_samples // seq_length
            for i, token in enumerate(label_seq):
                start = i * seg_len
                end = start + seg_len
                pattern = self.token_patterns[token]
                if pattern.shape[1] != seg_len:
                    pattern = F.interpolate(pattern.unsqueeze(0), size=seg_len, mode='linear', align_corners=False).squeeze(0)
                eeg[:, start:end] += pattern
            self.data.append(eeg)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


# ------------------------------
# Real EEG Dataset - loads data recorded by the T2TDataApp training application.
#
# CSV format produced by the app:
#   Metadata lines at top (key,value pairs), then a blank line, then:
#   event,word,word_id,trial,timestamp,relative_time_ms,ch1,...,ch16
#
# Event types relevant for training:
#   word_start  - marks the beginning of a word stimulus window
#   word_end    - marks the end of a word stimulus window
#   eeg_sample  - a single EEG sample row (16 channel values)
#
# This class extracts per-word EEG windows and resamples them to 256 samples.
class RealEEGDataset(Dataset):
    def __init__(self, csv_paths, channels=16, time_samples=256):
        """
        Args:
            csv_paths: list of CSV file paths from the training app
            channels: number of EEG channels (16 for Cyton+Daisy)
            time_samples: model input size (will resample to this length)
        """
        self.channels = channels
        self.time_samples = time_samples
        self.data = []
        self.labels = []

        for csv_path in csv_paths:
            self._load_csv(csv_path)

        print(f"RealEEGDataset: loaded {len(self.data)} word trials from {len(csv_paths)} file(s)")

    def _load_csv(self, csv_path):
        """Parse a single session CSV file and extract word EEG windows."""
        ch_cols = [f"ch{i}" for i in range(1, self.channels + 1)]

        # Step 1: Find the header row (starts with 'event,')
        header_line = 0
        with open(csv_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if line.startswith("event,"):
                    header_line = i
                    break

        # Step 2: Read the data section as a list of dicts
        rows = []
        with open(csv_path, "r", encoding="utf-8") as f:
            for _ in range(header_line):
                next(f)
            reader = csv.DictReader(f)
            for row in reader:
                rows.append(row)

        if not rows:
            print(f"  Warning: no data rows in {csv_path}")
            return

        # Step 3: Separate event markers from EEG samples
        word_starts = [r for r in rows if r.get("event") == "word_start"]
        word_ends   = [r for r in rows if r.get("event") == "word_end"]
        eeg_rows    = [r for r in rows if r.get("event") == "eeg_sample"]

        if not word_starts:
            print(f"  Warning: no word_start events in {csv_path}")
            return

        # Step 4: For each word_start event, find its matching word_end and
        #         collect the EEG samples in that time window.
        for ws in word_starts:
            word_name = ws.get("word", "").strip().upper()
            trial = ws.get("trial", "")
            t_start = float(ws.get("relative_time_ms", 0))

            # Find matching word_end (same word and trial)
            matching_ends = [
                we for we in word_ends
                if we.get("word", "").strip().upper() == word_name
                and we.get("trial", "") == trial
            ]
            if not matching_ends:
                continue
            t_end = float(matching_ends[0].get("relative_time_ms", 0))

            # Collect EEG samples within [t_start, t_end]
            window_rows = [
                r for r in eeg_rows
                if t_start <= float(r.get("relative_time_ms", 0)) <= t_end
            ]

            if len(window_rows) < 5:
                # Not enough EEG data in this window
                continue

            # Build (channels x n_samples) array
            try:
                eeg_array = np.array(
                    [[float(r.get(ch, 0) or 0) for ch in ch_cols] for r in window_rows],
                    dtype=np.float32
                ).T  # (16, n_samples)
            except ValueError:
                continue

            # Resample to self.time_samples using linear interpolation
            eeg_tensor = torch.tensor(eeg_array, dtype=torch.float).unsqueeze(0)  # (1, 16, n)
            if eeg_tensor.shape[2] != self.time_samples:
                eeg_tensor = F.interpolate(
                    eeg_tensor, size=self.time_samples, mode="linear", align_corners=False
                )
            eeg_tensor = eeg_tensor.squeeze(0).to(device)  # (16, 256)

            # Map word name to vocab index
            if word_name not in WORD_NAME_TO_ID:
                print(f"  Warning: unknown word '{word_name}' in {csv_path}")
                continue
            vocab_idx = WORD_NAME_TO_ID[word_name]

            self.data.append(eeg_tensor)
            self.labels.append(torch.tensor([vocab_idx], dtype=torch.long).to(device))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


def collate_fn(batch):
    eegs, label_seqs = zip(*batch)
    eegs = torch.stack(eegs)
    return eegs, label_seqs


# ------------------------------
# Synthesize EEG for testing (simulated data)
def synthesize_eeg(label_seq, token_patterns, channels=16, time_samples=256):
    eeg = torch.randn(channels, time_samples, device=device) * 0.1
    seq_length = len(label_seq)
    seg_len = time_samples // seq_length
    for i, token in enumerate(label_seq):
        start = i * seg_len
        end = start + seg_len
        pattern = token_patterns[token]
        if pattern.shape[1] != seg_len:
            pattern = F.interpolate(pattern.unsqueeze(0), size=seg_len, mode='linear', align_corners=False).squeeze(0)
        eeg[:, start:end] += pattern
    return eeg


# ------------------------------
# Training loop
def train_model(model, dataloader, optimizer, ce_loss_fn, num_epochs=50,
                teacher_forcing_prob=0.8, scheduled_decay=0.98, epsilon=0.3, online_lr=1e-4):
    total_samples = len(dataloader.dataset)
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        total_tokens = 0
        correct_count = 0
        for eegs, label_seqs in dataloader:
            eegs = eegs.to(device)
            optimizer.zero_grad()
            logits = model(eegs)  # (batch, 32, vocab_size)

            batch_targets = []
            for seq in label_seqs:
                seq_list = seq.tolist()
                padded = seq_list + [EOS_TOKEN] * (32 - len(seq_list))
                batch_targets.append(padded)
            targets = torch.tensor(batch_targets, dtype=torch.long, device=device)
            loss = ce_loss_fn(logits.view(-1, VOCAB_SIZE), targets.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * logits.size(0) * logits.size(1)
            total_tokens += logits.size(0) * logits.size(1)

            # Online update for misclassified samples
            for i, true_seq in enumerate(label_seqs):
                true_list = true_seq.tolist()
                decoded = epsilon_greedy_decode(logits[i:i+1], epsilon=epsilon,
                                               target_seq=true_list, max_len=len(true_list))
                if decoded == true_list:
                    correct_count += 1
                else:
                    optimizer.zero_grad()
                    sample_logits = model(eegs[i:i+1])
                    sample_target = torch.tensor(
                        [true_list + [EOS_TOKEN] * (32 - len(true_list))],
                        dtype=torch.long, device=device
                    )
                    sample_loss = ce_loss_fn(sample_logits.view(-1, VOCAB_SIZE), sample_target.view(-1))
                    orig_lr = optimizer.param_groups[0]['lr']
                    for group in optimizer.param_groups:
                        group['lr'] = online_lr
                    sample_loss.backward()
                    optimizer.step()
                    for group in optimizer.param_groups:
                        group['lr'] = orig_lr

        avg_loss = total_loss / total_tokens
        exact_match_acc = correct_count / total_samples * 100.0
        print(f"Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}, Exact Match Accuracy = {exact_match_acc:.2f}%")

        # Per-sample predictions
        model.eval()
        with torch.no_grad():
            print(f"  Epoch {epoch+1} sample predictions:")
            for idx in range(min(total_samples, 10)):  # Print up to 10 samples
                eeg_sample, label_seq = dataset[idx]
                eeg_sample = eeg_sample.unsqueeze(0).to(device)
                sample_logits = model(eeg_sample)
                pred_tokens = epsilon_greedy_decode(sample_logits, epsilon=0.0,
                                                    target_seq=label_seq.tolist(),
                                                    max_len=len(label_seq))
                pred_sentence = " ".join([VOCAB[t] for t in pred_tokens])
                true_sentence = " ".join([VOCAB[t] for t in label_seq.tolist()])
                print(f"    Sample {idx+1}: Pred: [{pred_sentence}] | True: [{true_sentence}]")
        model.train()
        teacher_forcing_prob *= scheduled_decay
    print("Training finished.")


# ------------------------------
# Dynamic checkpoint loader
def load_checkpoint_dynamic(model, optimizer, filename="model_checkpoint.pth"):
    if os.path.exists(filename):
        print("Loading checkpoint...")
        checkpoint = torch.load(filename, map_location=device, weights_only=True)
        model_state = checkpoint['model_state_dict']
        mismatch = False
        if model_state['classifier.weight'].shape[0] != model.classifier.weight.shape[0]:
            print("Vocabulary size mismatch detected. Reinitializing classifier layer...")
            mismatch = True
            model_state.pop('classifier.weight')
            model_state.pop('classifier.bias')
        model.load_state_dict(model_state, strict=False)
        if not mismatch:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        else:
            print("Optimizer state not loaded due to classifier mismatch.")
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming training from epoch {start_epoch}.")
        return start_epoch
    return 0


def save_checkpoint(model, optimizer, epoch, filename="model_checkpoint.pth"):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'vocab': VOCAB,
        'vocab_size': VOCAB_SIZE,
    }
    torch.save(checkpoint, filename)
    print(f"Checkpoint saved at epoch {epoch}.")


# ------------------------------
# Inference helper: predict word from a raw EEG segment
def predict_word(model, eeg_tensor):
    """
    Given a (16, N) EEG tensor, resample to 256 samples and predict the word.

    Args:
        model: trained EEGToTextModel
        eeg_tensor: torch.Tensor of shape (16, N) in microvolts

    Returns:
        predicted word string (e.g. "water")
    """
    model.eval()
    with torch.no_grad():
        if eeg_tensor.shape[1] != 256:
            eeg_tensor = F.interpolate(
                eeg_tensor.unsqueeze(0), size=256, mode='linear', align_corners=False
            ).squeeze(0)
        eeg_input = eeg_tensor.unsqueeze(0).to(device)
        logits = model(eeg_input)
        pred_tokens = epsilon_greedy_decode(logits, epsilon=0.0, max_len=1)
        if pred_tokens:
            return VOCAB[pred_tokens[0]]
        return "<blank>"


# ------------------------------
# Main entry point
def main():
    global dataset

    # --- Data Loading ---
    # Look for real session CSVs from the training app first
    real_csv_dir = "backend/data/sessions"
    csv_files = glob.glob(os.path.join(real_csv_dir, "session_*.csv"))

    if csv_files:
        print(f"Found {len(csv_files)} real session CSV(s). Loading real EEG data...")
        real_dataset = RealEEGDataset(csv_files)
        if len(real_dataset) > 0:
            dataset = real_dataset
            print(f"Using real EEG dataset: {len(dataset)} trials")
        else:
            print("Real dataset is empty (no valid word windows found). Using simulated data.")
            dataset = EEGDataset(num_samples=200)
    else:
        print("No real session CSVs found. Using simulated EEG data for pre-training.")
        print(f"  (To use real data, run the training app and save sessions to: {real_csv_dir})")
        dataset = EEGDataset(num_samples=200)

    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

    # --- Model ---
    model = EEGToTextModel(hidden_dim=128, vocab_size=VOCAB_SIZE, num_layers=2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=EOS_TOKEN)

    checkpoint_file = "model_checkpoint.pth"
    start_epoch = load_checkpoint_dynamic(model, optimizer, filename=checkpoint_file)

    total_epochs = 380
    for epoch in range(start_epoch, total_epochs):
        train_model(model, dataloader, optimizer, ce_loss_fn, num_epochs=1,
                    teacher_forcing_prob=0.8, scheduled_decay=0.98, epsilon=0.3, online_lr=1e-4)
        save_checkpoint(model, optimizer, epoch, filename=checkpoint_file)

    # --- Test on all 10 words ---
    print("\nTest Predictions on All 10 Target Words:")
    model.eval()
    with torch.no_grad():
        if hasattr(dataset, 'token_patterns'):
            # Simulated dataset - synthesize EEG for testing
            for target in new_target_sentences:
                test_eeg = synthesize_eeg(target, dataset.token_patterns)
                test_eeg = test_eeg.unsqueeze(0)
                test_logits = model(test_eeg)
                pred_tokens = epsilon_greedy_decode(test_logits, epsilon=0.0,
                                                    target_seq=target, max_len=1)
                pred_word = VOCAB[pred_tokens[0]] if pred_tokens else "<blank>"
                true_word = VOCAB[target[0]]
                print(f"  True: [{true_word}] -> Predicted: [{pred_word}]")
        else:
            # Real dataset - test on a subset of samples
            for idx in range(min(len(dataset), 20)):
                eeg_sample, label_seq = dataset[idx]
                pred_word = predict_word(model, eeg_sample)
                true_word = VOCAB[label_seq[0].item()]
                print(f"  Sample {idx+1} | True: [{true_word}] -> Predicted: [{pred_word}]")


if __name__ == "__main__":
    main()
